In [1]:
import os
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import glob
import time
import logging
import re
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LSTM, Bidirectional

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set MNE verbosity
mne.set_log_level('ERROR')

# Define dataset parameters
data_dir = r'C:\Users\arnna\Desktop\fypbilstm2\STEW Dataset'
fs = 128  # Sampling frequency in Hz
T = 150   # Duration in seconds
n_samples = int(T * fs)  # 19200 samples
ch_names = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
n_channels = len(ch_names)

# Data augmentation parameters
window_size = 4 * fs  # 4 seconds = 512 samples
step_size = 2 * fs    # 2 seconds = 256 samples (50% overlap)

# Step 1: Data Labeling (Ternary Classification)
def create_labels_file(data_dir):
    ratings_file = os.path.join(data_dir, 'ratings.txt')
    class_dict = {}
    with open(ratings_file, 'r') as f:
        for line in f:
            parts = [part.strip() for part in line.split(',')]
            if len(parts) == 3:
                try:
                    subject_id, rating_lo, rating_hi = map(int, parts)
                    subject_id_str = str(subject_id).zfill(2)
                    class_lo = 0 if 1 <= rating_lo <= 3 else (1 if 4 <= rating_lo <= 6 else 2)
                    class_hi = 0 if 1 <= rating_hi <= 3 else (1 if 4 <= rating_hi <= 6 else 2)
                    class_dict[(subject_id_str, 'lo')] = class_lo
                    class_dict[(subject_id_str, 'hi')] = class_hi
                    logger.info(f"Subject {subject_id_str}: class_lo = {class_lo}, class_hi = {class_hi}")
                except ValueError:
                    logger.warning(f"Could not convert line to integers: {line.strip()}")

    data_files = [f for f in os.listdir(data_dir) if f.endswith('.txt') and f != 'ratings.txt']
    mappings = []
    for file_name in data_files:
        if file_name.startswith('sub') and file_name.endswith('.txt'):
            subject_id_str = file_name[3:5]
            task = file_name[6:8]
            if (subject_id_str, task) in class_dict:
                class_label = class_dict[(subject_id_str, task)]
                mappings.append((file_name, class_label))
            else:
                logger.warning(f"No class label found for {file_name}")

    df = pd.DataFrame(mappings, columns=['file_name', 'class_label'])
    output_file = os.path.join(data_dir, 'labels.csv')
    df.to_csv(output_file, index=False)
    logger.info(f"Created labels.csv with {len(df)} entries.")
    return df

# Load EEG data with labels
def load_eeg_data(data_dir, labels_df):
    data = []
    labels = []
    subject_ids = []
    conditions = []
    delimiters = [r'\s+', '\t', ',', ';']
    for _, row in labels_df.iterrows():
        file_name = row['file_name']
        label = row['class_label']
        file_path = os.path.join(data_dir, file_name)
        file_size = os.path.getsize(file_path) / 1024  # Size in KB
        logger.info(f"Processing {file_name} (Size: {file_size:.2f} KB)")
        if file_size < 10:  # Skip very small files
            logger.warning(f"Skipping {file_name}: File too small ({file_size:.2f} KB)")
            continue
        for sep in delimiters:
            try:
                eeg_data = pd.read_csv(file_path, sep=sep, header=None, engine='python')
                logger.info(f"Parsed {file_name} with delimiter: {sep}")
                if eeg_data.shape[0] < int(0.9 * n_samples) or eeg_data.shape[1] != n_channels:
                    logger.warning(f"Skipping {file_name}: Incorrect shape {eeg_data.shape}, expected (~{n_samples}, {n_channels})")
                    break
                if not np.all(eeg_data.apply(lambda x: np.issubdtype(x.dtype, np.number))):
                    logger.warning(f"Skipping {file_name}: Contains non-numeric data")
                    break
                eeg_data = eeg_data.iloc[:n_samples].values.T  # Transpose to channels x samples
                if np.any(np.isnan(eeg_data)):
                    logger.warning(f"Skipping {file_name}: Contains NaN values")
                    continue
                data.append(eeg_data)
                labels.append(label)
                subject_id = file_name[3:5]
                condition = 'high' if 'hi' in file_name.lower() else 'low'
                subject_ids.append(subject_id)
                conditions.append(condition)
                logger.info(f"Loaded {file_name}: Shape {eeg_data.shape}, Label: {label}")
                break
            except Exception as e:
                logger.error(f"Error reading {file_name} with delimiter {sep}: {str(e)}")
                if sep == delimiters[-1]:
                    logger.warning(f"Skipping {file_name}: Failed to parse with all delimiters")
                    try:
                        eeg_data = pd.read_fwf(file_path, header=None)
                        if eeg_data.shape[0] < int(0.9 * n_samples) or eeg_data.shape[1] != n_channels:
                            logger.warning(f"Skipping {file_name}: Incorrect shape {eeg_data.shape} with fixed-width")
                            break
                        eeg_data = eeg_data.iloc[:n_samples].values.T
                        if np.any(np.isnan(eeg_data)):
                            logger.warning(f"Skipping {file_name}: Contains NaN values with fixed-width")
                            continue
                        data.append(eeg_data)
                        labels.append(label)
                        subject_id = file_name[3:5]
                        condition = 'high' if 'hi' in file_name.lower() else 'low'
                        subject_ids.append(subject_id)
                        conditions.append(condition)
                        logger.info(f"Loaded {file_name}: Shape {eeg_data.shape} with fixed-width")
                    except Exception as e:
                        logger.error(f"Error reading {file_name} with fixed-width: {str(e)}")
    if not data:
        logger.error("No valid EEG files were loaded.")
        raise ValueError("No valid EEG files were loaded.")
    return np.array(data), np.array(labels), subject_ids, conditions

# Step 2: Preprocessing
def bandpass_filter(data, lowcut=1.0, highcut=40.0, fs=fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='bandpass')
    filtered = signal.filtfilt(b, a, data, axis=-1)
    return filtered

def plot_eeg_channels(data, ch_names, fs, filename='eeg_channels_ml.png'):
    plt.figure(figsize=(15, 10))
    times = np.linspace(0, T, data.shape[1])
    for i, ch in enumerate(ch_names):
        plt.subplot(len(ch_names), 1, i+1)
        plt.plot(times, data[i], label=ch)
        plt.ylabel(ch)
        if i == len(ch_names) - 1:
            plt.xlabel('Time (s)')
        plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(data_dir, filename))
    plt.close()

# Data augmentation using sliding windows
def augment_data(data, labels, window_size, step_size):
    augmented_data = []
    augmented_labels = []
    for i in range(data.shape[0]):  # Iterate over each file
        eeg_data = data[i]  # Shape: (n_channels, n_samples)
        n_windows = int((eeg_data.shape[1] - window_size) / step_size) + 1
        for start in range(0, eeg_data.shape[1] - window_size + 1, int(step_size)):
            window = eeg_data[:, start:start + window_size]
            augmented_data.append(window)
            augmented_labels.append(labels[i])
    return np.array(augmented_data), np.array(augmented_labels)

# Create Bi-LSTM model (Ternary Classification)
def create_bilstm(input_shape):
    model = Sequential([
        # First Bi-LSTM layer
        Bidirectional(LSTM(128, return_sequences=True, input_shape=input_shape)),
        BatchNormalization(),
        Dropout(0.3),
        
        # Second Bi-LSTM layer
        Bidirectional(LSTM(64, return_sequences=False)),
        BatchNormalization(),
        Dropout(0.3),
        
        # Dense layers for classification
        Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dropout(0.4),
        Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dense(3, activation='softmax')  # Ternary classification (3 classes)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Main pipeline
def main():
    logger.info("Starting EEG data processing...")

    # Step 1: Data Labeling
    try:
        labels_df = create_labels_file(data_dir)
    except Exception as e:
        logger.error(f"Failed to create labels file: {str(e)}")
        return

    # Load and preprocess data
    try:
        data, labels, subject_ids, conditions = load_eeg_data(data_dir, labels_df)
    except Exception as e:
        logger.error(f"Failed to load data: {str(e)}")
        return
    logger.info(f"Loaded {len(data)} valid files.")

    start_time = time.time()
    all_preprocessed_dfs = []

    # Preprocess data (only bandpass filtering)
    for i, (eeg_data, subject_id, condition) in enumerate(zip(data, subject_ids, conditions)):
        # Apply bandpass filter
        filtered_data = bandpass_filter(eeg_data, lowcut=1.0, highcut=40.0)
        
        # Plot channels for the first file
        if i == 0:
            plot_eeg_channels(filtered_data, ch_names, fs)
            logger.info("Plotted channels for first file.")

        # Save bandpass filtered data for this file
        df = pd.DataFrame(filtered_data.T, columns=ch_names)
        df['Subject_ID'] = subject_id
        df['Condition'] = condition
        df['Label'] = labels[i]
        df['Sample_Index'] = range(len(df))
        all_preprocessed_dfs.append(df)
        logger.info(f"Preprocessed file {i+1} (Subject: {subject_id}, Condition: {condition})")

    # Save all bandpass filtered data to extracted.csv
    if all_preprocessed_dfs:
        all_preprocessed_df = pd.concat(all_preprocessed_dfs, ignore_index=True)
        all_preprocessed_df.to_csv(os.path.join(data_dir, 'extracted.csv'), index=False)
        logger.info(f"Saved preprocessed data for {len(all_preprocessed_dfs)} files to extracted.csv (Shape: {all_preprocessed_df.shape})")
    else:
        logger.error("No preprocessed data to save to extracted.csv.")
        return

    # Data augmentation
    X, y = augment_data(data, labels, window_size, step_size)
    logger.info(f"After augmentation, dataset size: {X.shape[0]} samples (shape: {X.shape})")

    # Reshape X to (n_samples, timesteps, channels) for LSTM
    X = X.transpose(0, 2, 1)  # From (n_samples, 14, 512) to (n_samples, 512, 14)
    logger.info(f"Reshaped X to: {X.shape}")

    # Convert labels to categorical (one-hot encoded) for ternary classification
    y_cat = tf.keras.utils.to_categorical(y, num_classes=3)

    # Standardize the data
    scaler = StandardScaler()
    X_reshaped = X.reshape(X.shape[0], -1)  # Flatten for scaling
    X_scaled = scaler.fit_transform(X_reshaped)
    X = X_scaled.reshape(X.shape)  # Reshape back to (n_samples, timesteps, channels)

    # 70/30 train-test split with stratification
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_cat, test_size=0.3, stratify=y, random_state=123
    )
    logger.info(f"Training set size: {X_train.shape[0]} samples")
    logger.info(f"Test set size: {X_test.shape[0]} samples")

    # Create and train Bi-LSTM model
    model = create_bilstm(input_shape=(window_size, n_channels))  # (timesteps, channels) = (512, 14)
    
    history = model.fit(
        X_train, y_train,
        epochs=200,
        batch_size=32,
        validation_data=(X_test, y_test),
        verbose=1
    )

    # Evaluate on test set
    test_pred = model.predict(X_test, verbose=0)
    test_pred_classes = np.argmax(test_pred, axis=1)
    test_true_classes = np.argmax(y_test, axis=1)
    test_accuracy = accuracy_score(test_true_classes, test_pred_classes)
    logger.info(f"Test Accuracy: {test_accuracy:.4f}")

    # Plot confusion matrix
    plt.figure(figsize=(5, 4))
    cm = confusion_matrix(test_true_classes, test_pred_classes, labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low (0)', 'Medium (1)', 'High (2)'], yticklabels=['Low (0)', 'Medium (1)', 'High (2)'])
    plt.title('Confusion Matrix - 70/30 Split')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.savefig(os.path.join(data_dir, 'confusion_matrix.png'))
    plt.close()

    # Plot validation loss
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(data_dir, 'validation_loss_ml.png'))
    plt.close()

    # Plot validation accuracy
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(data_dir, 'validation_accuracy_ml.png'))
    plt.close()

    # Plot validation loss and accuracy on the same graph
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    # Plot validation accuracy on the left y-axis
    color_acc = 'tab:blue'
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Validation Accuracy', color=color_acc)
    ax1.plot(range(1, len(history.history['val_accuracy']) + 1), history.history['val_accuracy'], label='Validation Accuracy', color=color_acc)
    ax1.tick_params(axis='y', labelcolor=color_acc)
    ax1.grid(True)

    # Create a second y-axis for validation loss
    ax2 = ax1.twinx()
    color_loss = 'tab:orange'
    ax2.set_ylabel('Validation Loss', color=color_loss)
    ax2.plot(range(1, len(history.history['val_loss']) + 1), history.history['val_loss'], label='Validation Loss', color=color_loss)
    ax2.tick_params(axis='y', labelcolor=color_loss)

    # Add title and legend
    plt.title('Validation Loss and Accuracy')
    fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
    plt.savefig(os.path.join(data_dir, 'avg_val_loss_accuracy.png'))
    plt.close()
    logger.info("Saved validation loss and accuracy plot: avg_val_loss_accuracy.png")

    logger.info(f"Total processing time: {time.time() - start_time:.2f} seconds")

if __name__ == "__main__":
    main()

2025-06-07 12:06:56,239 - INFO - Starting EEG data processing...
2025-06-07 12:06:56,252 - INFO - Subject 01: class_lo = 0, class_hi = 2
2025-06-07 12:06:56,252 - INFO - Subject 02: class_lo = 0, class_hi = 1
2025-06-07 12:06:56,254 - INFO - Subject 03: class_lo = 0, class_hi = 1
2025-06-07 12:06:56,255 - INFO - Subject 04: class_lo = 0, class_hi = 1
2025-06-07 12:06:56,256 - INFO - Subject 06: class_lo = 1, class_hi = 2
2025-06-07 12:06:56,256 - INFO - Subject 07: class_lo = 0, class_hi = 1
2025-06-07 12:06:56,257 - INFO - Subject 08: class_lo = 0, class_hi = 2
2025-06-07 12:06:56,260 - INFO - Subject 09: class_lo = 0, class_hi = 2
2025-06-07 12:06:56,260 - INFO - Subject 10: class_lo = 0, class_hi = 1
2025-06-07 12:06:56,261 - INFO - Subject 11: class_lo = 0, class_hi = 1
2025-06-07 12:06:56,263 - INFO - Subject 12: class_lo = 0, class_hi = 1
2025-06-07 12:06:56,264 - INFO - Subject 13: class_lo = 0, class_hi = 2
2025-06-07 12:06:56,264 - INFO - Subject 14: class_lo = 0, class_hi = 2

Epoch 1/200
146/146 [==============================] - 233s 1s/step - loss: 3.2997 - accuracy: 0.4069 - val_loss: 3.0558 - val_accuracy: 0.5405
Epoch 2/200
146/146 [==============================] - 210s 1s/step - loss: 3.0807 - accuracy: 0.4642 - val_loss: 2.9025 - val_accuracy: 0.5761
Epoch 3/200
146/146 [==============================] - 199s 1s/step - loss: 2.9238 - accuracy: 0.5004 - val_loss: 2.7526 - val_accuracy: 0.6156
Epoch 4/200
146/146 [==============================] - 202s 1s/step - loss: 2.8085 - accuracy: 0.5187 - val_loss: 2.6451 - val_accuracy: 0.6121
Epoch 5/200
146/146 [==============================] - 210s 1s/step - loss: 2.6793 - accuracy: 0.5541 - val_loss: 2.5245 - val_accuracy: 0.6361
Epoch 6/200
146/146 [==============================] - 195s 1s/step - loss: 2.5639 - accuracy: 0.5789 - val_loss: 2.4255 - val_accuracy: 0.6316
Epoch 7/200
146/146 [==============================] - 187s 1s/step - loss: 2.4584 - accuracy: 0.5867 - val_loss: 2.2766 - val_accuracy:

2025-06-07 22:14:42,684 - INFO - Test Accuracy: 0.8819
2025-06-07 22:14:43,278 - INFO - Saved validation loss and accuracy plot: avg_val_loss_accuracy.png
2025-06-07 22:14:43,278 - INFO - Total processing time: 36429.01 seconds
